# Checks — Workshop 2 — Delta Lake

Self-study helper for the static Workshop Guide and Databricks Free Edition. In class the Workshop Guide app runs the same checks with its **Check** button.

Load it in `WORKSHOP_delta` with `%run ./_checks/ws_delta` after the setup cells. It defines `check("ws_delta", "tNN")`, which looks at the table, its schema and history, and at the results your cells save (`<catalog>.default._ws_ws_delta_<step>`), prints `PASS — ...` or `FAIL — ...` with `_result()` and writes one row to the progress table with `record()`, both from `_progress`.

> **Note** Participants do not need to open or edit this notebook.

In [ ]:
%run ./_progress

In [ ]:
# Helpers shared by the Delta workshop checks
import datetime as _wsd_datetime
from pyspark.sql import functions as F


def _wsd_table():
    return f"{CATALOG}.{BRONZE_SCHEMA}.lab_delta_cust_ws"


def _wsd_need_table():
    assert spark.catalog.tableExists(_wsd_table()), \
        "Table lab_delta_cust_ws not found. Run the setup cell and step 01 first."
    return spark.table(_wsd_table())

In [ ]:
# Results saved by the workshop cells and the table history
def _wsd_saved(step, where):
    """Result saved by the last line of a cell: <catalog>.default._ws_ws_delta_<step>."""
    name = f"{CATALOG}.default._ws_ws_delta_{step}"
    assert spark.catalog.tableExists(name), \
        f"Table _ws_ws_delta_{step} not found. Run {where} to the end: its last line saves your result."
    return spark.table(name)


def _wsd_ops(after_version=-1):
    """Operations from DESCRIBE HISTORY, newest first, only versions > after_version."""
    rows = spark.sql(f"DESCRIBE HISTORY {_wsd_table()}").select("version", "operation").collect()
    return [r["operation"] for r in sorted(rows, key=lambda r: -r["version"]) if r["version"] > after_version]


def _wsd_no_where(params):
    """A DELETE without WHERE (the accident): no predicate, or ["true"] on newer runtimes."""
    p = ((params or {}).get("predicate") or "").replace(" ", "").lower()
    return p in ("", "[]", '["true"]')

In [ ]:
# t01: managed Delta table created from customers_csv (customers.csv in your Volume)
def _wsd_t01():
    df = _wsd_need_table()
    fmt = spark.sql(f"DESCRIBE DETAIL {_wsd_table()}").first()["format"]
    assert fmt == "delta", f"The table format is '{fmt}', expected 'delta'. Create it with CREATE TABLE ... AS SELECT, without USING."
    assert "customer_id" in df.columns, "Column customer_id is missing. Select all columns of the view with *."
    n = df.count()
    assert n > 0, "The table is empty. Load the rows with AS SELECT ... FROM customers_csv."
    src = spark.read.option("header", "true").csv(f"{DATASET_PATH}/customers/customers.csv").count()
    assert n == src, f"The table has {n:,} rows, customers_csv has {src:,}. Recreate it from the whole view with CREATE OR REPLACE."
    return f"lab_delta_cust_ws is a Delta table with {n:,} rows."

In [ ]:
# t02 (optional): schema enforcement rejected bad_df
def _wsd_t02():
    df = _wsd_need_table()
    assert "mood" not in df.columns, \
        "The table has a column mood: the append evolved the schema. Start again from the setup cell and append without mergeSchema."
    assert df.filter(F.col("customer_id") == "BAD001").count() == 0, \
        "Row BAD001 is in the table: the bad append was accepted. Append bad_df without any schema option."
    row = _wsd_saved("t02", "the step 02 cell").first()
    err = row["error"] if row else None
    assert err, \
        "enforcement_error is empty: the append did not run or did not fail. Replace pass with the append of bad_df to TABLE."
    assert "schema" in err.lower(), \
        f"The error must come from Delta (schema mismatch), not from a typo. Your error: {err[:120]}"
    return "Delta rejected bad_df (extra column mood); the table is unchanged."

In [ ]:
# t03: column loyalty_tier (STRING) added
def _wsd_t03():
    types = dict(_wsd_need_table().dtypes)
    assert "loyalty_tier" in types, "Column loyalty_tier not found. Add it with ALTER TABLE ... ADD COLUMN."
    assert types["loyalty_tier"] == "string", \
        f"loyalty_tier has type {types['loyalty_tier']}, expected STRING. Start again from the setup cell and add it as STRING."
    return "Column loyalty_tier (STRING) added; existing rows have NULL in it."

In [ ]:
# t04 (optional): CHECK constraint valid_country active and enforced
def _wsd_t04():
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {_wsd_table()}").collect()}
    expr = props.get("delta.constraints.valid_country")
    assert expr, "Constraint valid_country not found. Add it with ALTER TABLE ... ADD CONSTRAINT valid_country CHECK (...)."
    assert "country" in expr.lower() and "null" in expr.lower(), \
        f"valid_country checks '{expr}'. It must reject rows where country is NULL."
    assert _wsd_need_table().filter(F.col("customer_id") == "BAD_C").count() == 0, \
        "BAD_C is in the table, so the constraint did not stop it. Add the constraint before the INSERT."
    row = _wsd_saved("t04", "the step 04 cell").first()
    err = row["error"] if row else None
    assert err, \
        "constraint_error is empty. Insert customer BAD_C with country NULL inside the try block."
    assert "constraint" in err.lower() or "DELTA_VIOLATE_CONSTRAINT" in err, \
        f"The error must come from the Delta constraint, not from a typo. Your error: {err[:120]}"
    return f"Constraint valid_country ({expr}) rejected the row with a NULL country."

In [ ]:
# t05: three test customers inserted once
def _wsd_t05():
    rows = _wsd_need_table().filter(F.col("customer_id").startswith("TEST")).select("customer_id", "country").collect()
    ids = sorted(r["customer_id"] for r in rows)
    assert ids == ["TEST001", "TEST002", "TEST003"], \
        f"Expected TEST001, TEST002, TEST003 once each, found {ids or 'none'}. Inserted twice? " \
        "Run DELETE FROM lab_delta_cust_ws WHERE customer_id LIKE 'TEST%', then insert again."
    bad = [r["customer_id"] for r in rows if r["country"] != "TestLand"]
    assert not bad, f"country is not 'TestLand' for {bad}. Use the string 'TestLand' in quotes."
    assert "WRITE" in _wsd_ops()[:5], "No recent WRITE version in the history. Run your INSERT statement."
    return "TEST001, TEST002 and TEST003 inserted in one new table version."

In [ ]:
# t06: only TEST001 updated to Poland / Gold
def _wsd_t06():
    df = _wsd_need_table()
    rows = df.filter(F.col("customer_id") == "TEST001").collect()
    assert len(rows) == 1, f"TEST001 found {len(rows)} times, expected once. Run step 05 first."
    got = (rows[0]["country"], rows[0]["loyalty_tier"])
    assert got == ("Poland", "Gold"), f"TEST001 has country={got[0]}, loyalty_tier={got[1]}; expected Poland and Gold."
    gold = df.filter(F.col("loyalty_tier") == "Gold").count()
    assert gold == 1, f"{gold:,} rows have loyalty_tier Gold, expected 1. Limit the UPDATE with WHERE; start again from the setup cell."
    moved = df.filter(F.col("customer_id").isin("TEST002", "TEST003") & (F.col("country") != "TestLand")).count()
    assert moved == 0, "TEST002 or TEST003 changed too. Only TEST001 may change: check the WHERE condition."
    assert "UPDATE" in _wsd_ops()[:5], "No recent UPDATE version in the history. Run your UPDATE statement."
    return "TEST001 updated to Poland / Gold; no other row changed."

In [ ]:
# t07 (optional): TEST002 and TEST003 deleted, TEST001 kept
def _wsd_t07():
    rows = _wsd_need_table().filter(F.col("customer_id").startswith("TEST")).select("customer_id").collect()
    ids = {r["customer_id"] for r in rows}
    left = sorted(ids & {"TEST002", "TEST003"})
    assert not left, f"Still in the table: {left}. Delete both with one DELETE ... WHERE customer_id IN (...)."
    assert "TEST001" in ids, "TEST001 is missing: the DELETE removed too much. Run steps 05 and 06 again, then delete only TEST002 and TEST003."
    assert "DELETE" in _wsd_ops()[:5], "No recent DELETE version in the history. Run your DELETE statement."
    return "TEST002 and TEST003 deleted; TEST001 kept."

In [ ]:
# t08: MERGE updated 2 customers and inserted 1, nothing else removed (the batch of updates_view, fixed values)
_WSD_T08_BATCH = [("TEST001", F.col("customer_id") == "TEST001", "MergedAlice", "Platinum"),
                  ("The existing customer from updates_view", F.col("first_name") == "MergedProd", "MergedProd", "Silver"),
                  ("MERGE_NEW", F.col("customer_id") == "MERGE_NEW", "NewViaMerge", None)]


def _wsd_t08():
    df = _wsd_need_table()
    for label, cond, first_name, tier in _WSD_T08_BATCH:
        rows = df.filter(cond).select("first_name", "loyalty_tier").collect()
        assert rows or label != _WSD_T08_BATCH[1][0], f"{label} does not have the values from updates_view. Check WHEN MATCHED ... UPDATE SET."
        assert len(rows) == 1, f"{label} found {len(rows)} times, expected once. Check ON and WHEN NOT MATCHED ... INSERT."
        assert (rows[0]["first_name"], rows[0]["loyalty_tier"]) == (first_name, tier), \
            f"{label} does not have the values from updates_view. Check WHEN MATCHED ... UPDATE SET."
    others = df.filter(~F.col("customer_id").isin("TEST001", "MERGE_NEW") & ~F.col("first_name").eqNullSafe("MergedProd")).count()
    assert others > 0, "Only the batch rows are left: a NOT MATCHED BY SOURCE ... DELETE removed the others. Start again from the setup cell."
    assert "MERGE" in _wsd_ops()[:5], "No recent MERGE version in the history. Run your MERGE statement."
    return f"MERGE applied: {len(_WSD_T08_BATCH)} batch rows match updates_view, the other {others:,} customers are kept."

In [ ]:
# t09 (optional): full_history holds DESCRIBE HISTORY
def _wsd_t09():
    fh = _wsd_saved("t09", "your step 09 cell")
    assert "operation" in fh.columns, "full_history has no operation column, so it is not the result of DESCRIBE HISTORY."
    ops = [r["operation"] for r in fh.select("operation").collect()]
    missing = [o for o in ("UPDATE", "MERGE") if o not in ops]
    assert not missing, f"{missing} not found in full_history. Run steps 06 and 08, then run step 09 again."
    return f"full_history has {len(ops)} versions, including UPDATE and MERGE."

In [ ]:
# t10: customers_v0 is version 0 of the table
def _wsd_t10():
    v0 = _wsd_saved("t10", "your step 10 cell (end your statement with ;)")
    cols = v0.columns
    assert "customer_id" in cols, "customers_v0 has no customer_id column. Select all columns with *."
    assert "loyalty_tier" not in cols, "customers_v0 has loyalty_tier, so it is not version 0 (the column came in step 03)."
    assert v0.filter(F.col("customer_id") == "TEST001").count() == 0, \
        "customers_v0 contains TEST001 (inserted in step 05), so it is not version 0."
    n0, n = v0.count(), _wsd_need_table().count()
    return f"customers_v0 is version 0: {n0:,} rows, no loyalty_tier (current table: {n:,} rows)."

In [ ]:
# t11 (optional): df_ts is the table at the time of the INSERT from step 05
def _wsd_t11():
    df = _wsd_saved("t11", "your step 11 cell")
    assert "customer_id" in df.columns and "country" in df.columns, \
        "df_ts must be TABLE read with TIMESTAMP AS OF ts_str (all columns)."
    rows = df.filter(F.col("customer_id") == "TEST001").select("country").collect()
    assert len(rows) == 1, "df_ts has no TEST001, so ts_insert is earlier than the INSERT of step 05."
    assert rows[0]["country"] == "TestLand", \
        f"TEST001 has country {rows[0]['country']} in df_ts, expected TestLand. Use the exact INSERT time, with microseconds."
    return "df_ts shows the table at the INSERT of step 05: TEST001 still has country TestLand."

In [ ]:
# t12: accidental DELETE (no WHERE) undone with RESTORE; the safe state comes from the history
def _wsd_t12():
    df = _wsd_need_table()
    h = spark.sql(f"DESCRIBE HISTORY {_wsd_table()}").select("version", "operation", "operationParameters").collect()
    h = sorted(h, key=lambda r: -r["version"])
    full = [r for r in h if r["operation"] == "DELETE" and _wsd_no_where(r["operationParameters"])]
    assert full, "No DELETE after safe_version. Run the accident first: DELETE FROM without WHERE."
    accident = full[0]["version"]
    restore = [r["version"] for r in h if r["operation"] == "RESTORE" and r["version"] > accident]
    assert restore, "No RESTORE after safe_version. Roll the table back to safe_version with RESTORE."
    assert not [r for r in h if r["operation"] == "DELETE" and r["version"] > restore[0]], \
        "The newest RESTORE is older than the DELETE. Run RESTORE after the accident."
    before = spark.sql(f"SELECT count(*) AS n FROM {_wsd_table()} VERSION AS OF {accident - 1}").first()["n"]
    assert before > 0, "The table was already empty before the accident. Restore to the version before the DELETE (see DESCRIBE HISTORY), then run the ready cell again."
    n = df.count()
    assert n == before, f"The table has {n:,} rows, expected {before:,} as before the accident. Restore to safe_version."
    return f"{n:,} rows are back; DELETE and RESTORE are in the history."

In [ ]:
# Step id -> (step type, check function)
_WSD_STEPS = {
    "t01": ("fill", _wsd_t01), "t02": ("code", _wsd_t02), "t03": ("fill", _wsd_t03),
    "t04": ("code", _wsd_t04), "t05": ("fill", _wsd_t05), "t06": ("fill", _wsd_t06),
    "t07": ("code", _wsd_t07), "t08": ("order", _wsd_t08), "t09": ("code", _wsd_t09),
    "t10": ("order", _wsd_t10), "t11": ("code", _wsd_t11), "t12": ("order", _wsd_t12),
}

In [ ]:
# check("ws_delta", "tNN"): called from the check cell of every step
def check(workshop, step):
    """Run the hidden checks for one Delta workshop step."""
    if workshop != "ws_delta" or step not in _WSD_STEPS:
        _result(False, f'unknown step {workshop}/{step}. Use check("ws_delta", "t01") ... check("ws_delta", "t12").')
        return
    step_type, fn = _WSD_STEPS[step]
    try:
        ok, msg = True, fn()
    except AssertionError as e:
        ok, msg = False, str(e)
    except Exception as e:  # Spark or Python error while inspecting the participant's state
        first = (str(e).strip().splitlines() or [type(e).__name__])[0][:120]
        ok, msg = False, f"{type(e).__name__}: {first}. Re-run the cells of this step, then this check."
    _result(ok, f"Step {step[1:]}: {msg}")
    record(workshop, step, step_type, ok, msg[:200])